In [2]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, count

def main():
    # Create Spark session with local master and file system configuration
    spark = (
        SparkSession.builder
        .appName("read_gold_dim_products")
        .master("local[*]")
        .config("spark.hadoop.fs.defaultFS", "file:///")
        .getOrCreate()
    )
    
    # Define the path to the gold dimension table
    gold_path = "file:/data/gold/dim_products"
    
    # Print header information
    print("Reading Gold dimension: dim_products")
    print(f"Path: {gold_path}")
    
    # Read the parquet files from gold layer
    df = spark.read.parquet(gold_path)
    
    # Display the schema of the dimension table
    print("\nSchema:")
    df.printSchema()
    
    # Show sample data from all records
    print("\nSample data:")
    df.show(truncate=False)
    
    # Filter and display only current records (is_current = true)
    print("\nCurrent records only (is_current = true):")
    df_current = df.filter(col("is_current") == True)
    df_current.show(truncate=False)
    
    # Count versions per product to identify products with multiple versions
    print("\nVersions per product:")
    (
        df.groupBy("product_code")
        .agg(count("*").alias("versions"))
        .orderBy(col("versions").desc())
        .show()
    )
    
    # Create a temporary view for SQL queries
    print("\nCreating temp view: dim_products")
    df.createOrReplaceTempView("dim_products")
    
    # Query current dimension records using SQL
    print("\nQuerying current dimension records:")
    spark.sql("""
        SELECT
            product_sk,
            product_code,
            product_description,
            sale_value,
            is_active,
            category_code,
            valid_from,
            valid_to
        FROM dim_products
        WHERE is_current = true
        ORDER BY product_code
    """).show(truncate=False)
    
    # Optional: Show historical changes for products with multiple versions
    print("\nProducts with historical changes:")
    spark.sql("""
        SELECT
            product_code,
            COUNT(*) as version_count
        FROM dim_products
        GROUP BY product_code
        HAVING COUNT(*) > 1
        ORDER BY version_count DESC
    """).show()
    
    # Optional: Show full history of a specific product (example with product_code = 1)
    print("\nFull history example (if product has versions):")
    spark.sql("""
        SELECT
            product_sk,
            product_code,
            product_description,
            sale_value,
            is_active,
            category_code,
            valid_from,
            valid_to,
            is_current
        FROM dim_products
        WHERE product_code = 1
        ORDER BY valid_from
    """).show(truncate=False)
    
    # Stop the Spark session
    spark.stop()

if __name__ == "__main__":
    main()

Reading Gold dimension: dim_products
Path: file:/data/gold/dim_products

Schema:
root
 |-- product_code: integer (nullable = true)
 |-- product_description: string (nullable = true)
 |-- sale_value: decimal(18,2) (nullable = true)
 |-- is_active: integer (nullable = true)
 |-- category_code: integer (nullable = true)
 |-- product_sk: integer (nullable = true)
 |-- valid_from: date (nullable = true)
 |-- valid_to: date (nullable = true)
 |-- is_current: boolean (nullable = true)


Sample data:
+------------+---------------------------+----------+---------+-------------+----------+----------+----------+----------+
|product_code|product_description        |sale_value|is_active|category_code|product_sk|valid_from|valid_to  |is_current|
+------------+---------------------------+----------+---------+-------------+----------+----------+----------+----------+
|1           |Clean Code Book            |45.90     |1        |1            |0         |2026-02-16|2026-02-16|false     |
|2           |